<a href="https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

# Load token and connect
HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN is None:
    raise ValueError("🚨 HF_TOKEN nahi mila! Left menu (🔑) se access ON karein.")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")
print("✅ DuckDB connection & HF token ready!")

✅ DuckDB connection & HF token ready!


## 1. Two paper findings + my methodology questions



**Finding 1:** The paper asserts that their model identifies the most valuable content clusters with extremely high precision.
* **My Methodology Question:** How exactly is "value" defined here? If the label relies on the same metrics used as inputs, does this create a risk of data leakage? I would like to see a clear separation between the input features and the ground truth label.

**Finding 2:** The model's performance was stated to be universally strong across all tested data.
* **My Methodology Question:** Does the validation design fully support this? Was the test set split randomly, or did they use a grouped/time-aware split? A random split might just measure the model's ability to memorize specific client baselines rather than generalize to new, unseen scenarios.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)


In my previous build, I used a standard random split. However, a random split can lead to data leakage if URLs from the same client are in both the training and testing sets (the model just memorizes that client's baseline).

To make the evaluation honest, I am applying a **Grouped Split by `client_hash_id`**. This ensures the model is tested on completely unseen clients. The table below compares the **measured** error before and after this honest adjustment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Load data with client_hash_id
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FACT_FEB = f"{FACT}/month=2026-02/*.parquet"

query = f"""
SELECT
    client_hash_id,
    content_hash_id as url_id,
    gsc_impressions,
    gsc_avg_position,
    gsc_clicks
FROM read_parquet('{FACT_FEB}')
WHERE gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
"""
df = con.execute(query).df().fillna(0)

features = ['gsc_impressions', 'gsc_avg_position']
X = df[features]
y = df['gsc_clicks']
groups = df['client_hash_id']

# --- BEFORE: Random Split (Week 5 Baseline) ---
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.2, random_state=42)
rf_rand = RandomForestRegressor(n_estimators=100, random_state=42)
rf_rand.fit(X_train_rand, y_train_rand)
rmse_rand = np.sqrt(mean_squared_error(y_test_rand, rf_rand.predict(X_test_rand)))

# --- AFTER: Honest Split (Grouped by Client) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestRegressor(n_estimators=100, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)
rmse_grp = np.sqrt(mean_squared_error(y_test_grp, rf_grp.predict(X_test_grp)))

# Compare Results
comparison_df = pd.DataFrame({
    "Split Design": ["Random Split (Over-optimistic)", "Grouped by Client (Honest)"],
    "Measured RMSE (Lower is better)": [rmse_rand, rmse_grp]
})
display(comparison_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Split Design,Measured RMSE (Lower is better)
0,Random Split (Over-optimistic),0.921742
1,Grouped by Client (Honest),0.874803


## 3. Leakage audit

I audited the features (`gsc_impressions`, `gsc_avg_position`) against the target (`gsc_clicks`).
* **The Risk:** Impressions and clicks are recorded in the exact same snapshot. Impressions directly limit the maximum possible clicks, which is a form of synchronous correlation.
* **Conclusion:** Because this model is built as a **directional** tool to rank existing/past content opportunities (and not to predict future traffic for un-indexed pages), this overlap is acceptable. However, it means the model's accuracy shouldn't be overstated.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


correlation_matrix = df[['gsc_impressions', 'gsc_avg_position', 'gsc_clicks']].corr()
print("Correlation with Target (gsc_clicks):")
print(correlation_matrix['gsc_clicks'])

Correlation with Target (gsc_clicks):
gsc_impressions     0.547916
gsc_avg_position   -0.065397
gsc_clicks          1.000000
Name: gsc_clicks, dtype: float64


## 4. Claim rewrite


**Original Thought:** "My model perfectly predicts the exact number of clicks every URL will get, making the old rule obsolete."

**Rewritten (Honest) Claim:** "The Random Forest model offers a **directional** improvement over the static baseline rule. Based on the **observed** data using a grouped split, the **measured** error is slightly lower. This model serves as a strong **decision-support** tool to help prioritize SEO efforts, rather than an absolute predictor of future traffic."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.